In [ ]:
pip install -q -U google-genai

In [ ]:
from google.colab import userdata
import google.generativeai as genai
import pandas as pd
import numpy as np
import os



## SETUP

In [ ]:
REQ_FOLDER = "/content/dataset/"
GT_FOLDER = "/content/human_tests/"
PROMPTS_FOLDER = "/content/prompts/"

PATH_REQ_BOOKMARK = os.path.join(REQ_FOLDER, "req_bookmark.txt")
PATH_REQ_HISTORY = os.path.join(REQ_FOLDER, "req_history.txt")
PATH_REQ_PASSWORD = os.path.join(REQ_FOLDER, "req_password.txt")

def read_txt_file(path):
    with open(path) as f:
        output = f.read()
    return output

REQ_BOOKMARK = read_txt_file(PATH_REQ_BOOKMARK)
REQ_HISTORY = read_txt_file(PATH_REQ_HISTORY)
REQ_PASSWORD = read_txt_file(PATH_REQ_PASSWORD)

### Expected Results

In [ ]:
#First case is used to examples
df_bookmarks = pd.read_csv(os.path.join(GT_FOLDER, "req_tests_bookmark.csv"))
df_history = pd.read_csv(os.path.join(GT_FOLDER, "req_tests_history.csv"))
df_passwords = pd.read_csv(os.path.join(GT_FOLDER, "req_tests_password.csv"))

df_history.dropna(inplace = True)
df_passwords.dropna(inplace = True)


bookmark_example = df_bookmarks.iloc[0]
history_example = df_history.iloc[0]
password_example = df_passwords.iloc[0]



df_bookmarks.drop(df_bookmarks.index[0], inplace=True)
df_history.drop(df_history.index[0], inplace=True)
df_passwords.drop(df_passwords.index[0], inplace=True)


### Prompts

In [ ]:
specializations = {"BOOKMARK": None, "HISTORY": None, "PASSWORD": None}

In [ ]:
specializations["BOOKMARK"] = f"""You are a Senior Software Testing Engineer.
Your objective is to generate technical, step-by-step test cases (pseudocode) for specific functionalities.
You must base your output strictly on the provided requirements document and the individual test descriptions,
generating a comprehensive step-by-step procedure for each description.

=== REQUIREMENTS DOCUMENT (KNOWLEDGE BASE) ===
{REQ_BOOKMARK}
==============================================

OUTPUT FORMAT:
Every generated test case must strictly follow this four-section structure. Do not add introductory or concluding remarks outside of this structure.

1. Purpose
2. Initial Conditions
3. Steps/Description
4. Expected Results

EXPECTED TEST EXAMPLE:

DESCRIPTION: {bookmark_example["test-overview"]}

EXPECTED OUTPUT:

{bookmark_example["test-description"]}
"""

In [ ]:
specializations["HISTORY"] = f"""You are a Senior Software Testing Engineer.
Your objective is to generate technical, step-by-step test cases (pseudocode) for specific functionalities.
You must base your output strictly on the provided requirements document and the individual test descriptions,
generating a comprehensive step-by-step procedure for each description.

=== REQUIREMENTS DOCUMENT (KNOWLEDGE BASE) ===
{REQ_HISTORY}
==============================================

OUTPUT FORMAT:
Every generated test case must strictly follow this four-section structure. Do not add introductory or concluding remarks outside of this structure.

1. Purpose
2. Initial Conditions
3. Steps/Description
4. Expected Results

EXPECTED TEST EXAMPLE:

DESCRIPTION: {history_example['test-overview']}

EXPECTED OUTPUT:

{history_example['test-description']}
"""

In [ ]:
specializations["PASSWORD"] = f"""You are a Senior Software Testing Engineer.
Your objective is to generate technical, step-by-step test cases (pseudocode) for specific functionalities.
You must base your output strictly on the provided requirements document and the individual test descriptions, generating a comprehensive step-by-step procedure for each description.

=== REQUIREMENTS DOCUMENT (KNOWLEDGE BASE) ===
{REQ_PASSWORD}
==============================================

OUTPUT FORMAT:
Every generated test case must strictly follow this four-section structure. Do not add introductory or concluding remarks outside of this structure.

1. Purpose
2. Initial Conditions
3. Steps/Description
4. Expected Results

EXPECTED TEST EXAMPLE:

DESCRIPTION: {password_example['test-overview']}

EXPECTED OUTPUT:

{password_example['test-description']}

"""

print(specializations["PASSWORD"])

### AI SETUP

In [ ]:
novo = userdata.get('GEMINI_TOKEN')
specialisation = ""
genai.configure(api_key=novo)
config = genai.GenerationConfig(
    temperature=0.6
)

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash",
    generation_config=config,
    system_instruction=specialisation
)

## EMBEDDINGS

In [ ]:
def get_embedding(phrase : str):
    embedding_structure = genai.embed_content(
        model = "models/gemini-embedding-2",
        content = phrase,
        task_type="semantic_similarity"
    )
    return embedding_structure['embedding']

def cossine_correlation(vectorA, vectorB):

    vectorA = np.array(vectorA)
    vectorB = np.array(vectorB)

    # Cossine = (Va dot Vb) / (mod(Va) * mod(Vb))
    dot = np.dot(vectorA, vectorB)
    norm1 = np.linalg.norm(vectorA)
    norm2 = np.linalg.norm(vectorB)

    return dot / (norm1 * norm2)


## GENERATING ANSWERS